# OGGM glacier grids and DEM GeoTIFFs

First step of the glacier grid pipeline (1.2 → 1.3 → 1.4). For every RGI region in `RGI_REGIONS` (`scripts/config_TF_Europe.py`):
1. Download the preprocessed OGGM glacier directories (OGGM v1.6, `elev_bands_w_data`) and export each glacier's gridded data (topography, aspect, slope, glacier mask, ...) to `OGGM/rgi_region_XX/xr_grids/<RGIId>.zarr`.
2. Export each glacier's DEM (`topo`) as a GeoTIFF in its projected CRS to `RGI_v6/<region folder>/geotiff/<RGIId>.tif`.

The GeoTIFFs are the input of `1.3 Create_skyview_factor.ipynb`, which computes the sky-view factor. Both the OGGM grids and the SVF are then combined in `1.4 Create_grids.ipynb`.

## Setting up:

In [ ]:
import os

# Silence KeOps (imported indirectly via the MBM/torch stack)
os.environ["PYKEOPS_VERBOSE"] = "0"
os.environ["KEOPS_VERBOSE"] = "0"

In [ ]:
import torch

# Force CUDA re-initialisation
if torch.cuda.is_available():
    torch.cuda.init()

print(torch.cuda.device_count())  # should now be > 0

In [ ]:
# --- System & utilities ---
import os
import sys
import warnings
from tqdm.notebook import tqdm

# Add repo root for MBM imports
sys.path.append(os.path.join(os.getcwd(), "../../"))

# --- Data science stack ---
import matplotlib.pyplot as plt

# --- Custom MBM modules ---
import massbalancemachine as mbm

# --- Warnings & autoreload (notebook) ---
warnings.filterwarnings("ignore")
%load_ext autoreload
%autoreload 2


# --- Configuration ---
# cfg.dataPath is the data root; RGI_REGIONS comes from config_TF_Europe
cfg = mbm.EuropeConfig()

from papers.Transformer_TF.scripts.config_TF_Europe import *
from papers.Transformer_TF.scripts.oggm import *
from papers.Transformer_TF.scripts.geodata import *

# Plot styles:
mbm.utils.seed_all(cfg.seed)
mbm.plots.use_mbm_style()

print("Using seed:", cfg.seed)

if torch.cuda.is_available():
    print("CUDA is available")
    mbm.utils.free_up_cuda()
else:
    print("CUDA is NOT available")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## Initialize OGGM directories:

In [ ]:
# Diagnostic helpers (not called below): summarise the table of glaciers with
# missing OGGM variables that export_oggm_grids returns.
def rgi_outline_path_from_spec(cfg, spec: dict) -> str:
    shp_rel = Path("RGI_v6") / spec["folder"] / spec["file"]
    return str(Path(cfg.dataPath) / shp_rel)


def plot_missing_for_one_region(
    df_missing,
    cfg,
    region_id: str,
    region_spec: dict,
    epsg_area: int = 3035,
):
    """
    df_missing columns expected:
      - rgi_id (string)
      - missing_vars (list-like)
    """
    if df_missing is None or len(df_missing) == 0:
        print(
            f"[{region_id} {region_spec.get('name','')}] No missing glaciers.")
        return

    # Load outlines for THIS region only
    shp_abs = rgi_outline_path_from_spec(cfg, region_spec)
    gdf = gpd.read_file(shp_abs).to_crs(epsg_area).rename(
        columns={"RGIId": "rgi_id"})

    gdf["area_km2"] = gdf.geometry.area / 1e6
    total_area = gdf["area_km2"].sum()

    # Merge areas onto missing table
    df_reg = df_missing.merge(gdf[["rgi_id", "area_km2"]],
                              on="rgi_id",
                              how="left")

    # Total missing area
    total_missing_area_km2 = df_reg["area_km2"].sum()
    total_missing_area_pct = (total_missing_area_km2 /
                              total_area) * 100 if total_area else float("nan")

    # Explode vars
    df_exploded = df_reg.explode("missing_vars")

    counts_missing_per_var = (
        df_exploded.groupby("missing_vars")["rgi_id"].nunique().sort_values(
            ascending=False))

    area_missing_per_var = (
        df_exploded.groupby("missing_vars")["area_km2"].sum().sort_values(
            ascending=False))
    perc_missing_per_var = (area_missing_per_var / total_area
                            ) * 100 if total_area else area_missing_per_var * 0

    # Print
    region_label = f"{region_id} {region_spec.get('name','')}".strip()
    print(f"\n[{region_label}]")
    print(f"Total glacier area with ANY missing variable: "
          f"{total_missing_area_km2:,.2f} km² ({total_missing_area_pct:.2f}%)")

    print("% of total glacier area missing per variable:")
    for var, pct in perc_missing_per_var.items():
        print(f"  - {var}: {pct:.2f}%")

    # Plot
    plt.figure(figsize=(7, 4))
    plt.bar(counts_missing_per_var.index.astype(str),
            counts_missing_per_var.values)
    plt.xlabel("Missing variable")
    plt.ylabel("Number of glaciers")
    plt.title(f"{region_label} – Count of glaciers missing each variable")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()

In [ ]:
# Optional: restrict processing to glaciers with stake data (currently unused,
# all glaciers of each region are processed).
# # Read rgis for all stakes:
# rgi_path = os.path.join(cfg.dataPath, "WGMS/RGIids_stakes_all_regions.csv")
# rgi_ids_all = pd.read_csv(rgi_path, dtype={"RGI_REGION": str})

# # concatenate with HMA:
# # Read rgis for all stakes:
# rgi_path_HMA = os.path.join(cfg.dataPath, "WGMS/RGIids_stakes_HMA.csv")
# rgi_ids_HMA = pd.read_csv(rgi_path_HMA, dtype={"RGI_REGION": str})
# rgi_ids_all = pd.concat([rgi_ids_all, rgi_ids_HMA], ignore_index=True)
# rgi_ids_all.groupby("RGI_REGION")["RGIId"].nunique()

In [ ]:
# Regions that will be processed
RGI_REGIONS

In [ ]:
stakes_only = []
# For each RGI region: download the OGGM glacier directories and export their
# gridded data to OGGM/rgi_region_XX/xr_grids/<RGIId>.zarr (one zarr per glacier)
for rgi_id, spec in RGI_REGIONS.items():
    print(f"Processing glacier region id: {rgi_id}...")
    # if rgi_id in stakes_only:
    #     rgi_list_region = rgi_ids_all[rgi_ids_all["RGI_REGION"] ==
    #                                   rgi_id]["RGIId"].tolist()
    # else:
    rgi_list_region = None  # None = all glaciers in the region

    gdirs, rgi_ids_list = initialize_oggm_glacier_directories(
        cfg,
        rgi_region=rgi_id,
        rgi_version="62",
        base_url=
        "https://cluster.klima.uni-bremen.de/~oggm/gdirs/oggm_v1.6/L1-L2_files/2025.6/elev_bands_w_data/",
        log_level="WARNING",
        task_list=None,
        rgi_ids_list=rgi_list_region)

    export_oggm_grids(cfg,
                      gdirs,
                      rgi_region=rgi_id,
                      subset_rgis=rgi_list_region)

## Export GeoTIFFs of DEMs:

Writes the DEM (`topo`) of each glacier grid as a GeoTIFF in its projected (metric) CRS. This is the input for the SVF computation in `1.3 Create_skyview_factor.ipynb`. The `geotiff/` folder of each region is emptied first.

In [ ]:
# Regions to restrict to stake glaciers (e.g. Alaska, W. Canada & US, Greenland); empty = all glaciers
STAKE_ONLY_REGIONS = {}

RUN = True
if RUN:
    for rgi_region, spec in RGI_REGIONS.items():
        rgi_region = str(rgi_region).zfill(2)
        # Input: OGGM zarr grids from the cell above. Output: one GeoTIFF per glacier
        path_RGIs = Path(
            cfg.dataPath) / f"OGGM/rgi_region_{rgi_region}" / "xr_grids"
        path_geotiff = Path(
            cfg.dataPath) / "RGI_v6" / spec["folder"] / "geotiff"

        if not path_RGIs.is_dir():
            print(
                f"Skipping RGI {rgi_region}: xr_grids not found: {path_RGIs}")
            continue

        # Get allowed RGIs for stake-only regions
        # if rgi_region in STAKE_ONLY_REGIONS:
        #     rgi_list_region = rgi_ids_all[rgi_ids_all["RGI_REGION"] ==
        #                                   rgi_region]["RGIId"].tolist()
        #     allowed_rgis = rgi_list_region
        #     print(
        #         f"[{rgi_region}] Filtering to {len(allowed_rgis)} stake RGIs.")
        # NOTE: since the stake-only `if` above is commented out, this `else` belongs
        # to `if not path_RGIs.is_dir()`, so allowed_rgis is always None.
        else:
            allowed_rgis = None

        # Find all glaciers saved as *.zarr and filter if needed
        zarr_paths = sorted(path_RGIs.glob("*.zarr"))
        if allowed_rgis is not None:
            zarr_paths = [zp for zp in zarr_paths if zp.stem in allowed_rgis]

        print(
            f"\nRGI {rgi_region} ({spec.get('name', '')}): processing {len(zarr_paths)} glacier zarr stores"
        )

        emptyfolder(str(path_geotiff))  # remove GeoTIFFs from a previous run
        for zp in tqdm(zarr_paths, desc=f"Export DEMs RGI {rgi_region}"):
            rgi_gl = zp.stem  # e.g. "RGI60-06.00568"
            try:
                export_glacier_dems_to_geotiff(str(path_RGIs),
                                               rgi_gl,
                                               str(path_geotiff),
                                               verbose=False)
            except ValueError as e:  # e.g. missing DEM or CRS
                print(f"Skipping {rgi_gl}: {e}")
                continue